In [ ]:
import copy
import gc

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from its.search import InverseTransformationSearch
from search.parallel_gradient import ParallelGradientDescent
from utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]
budget = 60

In [ ]:
#NOTE already rerun for using the whole embedding cache

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
from utils.eval.vis import vis_dataset
batch_size = next(iter(train_loader))[0].shape[0]
vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from experiment_thesis.main import train_and_get_model,train_or_load_energy_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "uncertainty")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]


In [ ]:
from utils.transforms.apply import grid_resample
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="sobol"
    ).to(device)

In [ ]:
from embedding_cache import LayerEmbeddingCache



cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

from torch.utils.data import SequentialSampler
import embedding_cache
import importlib
importlib.reload(embedding_cache)
from embedding_cache import LayerEmbeddingCache
from confidence.input_transform import RandomProjectionModule


transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"
from experiment_thesis.dataset_preperation.get_dataset import get_layer_embedding_cache_config,create_layer_embedding_cache
cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)

In [ ]:
cache_config

In [ ]:
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)


In [ ]:
from search.shgo import SHGO
random_search  = SHGO(initial_samples=60,local_runs=1,local_max_steps=0)

In [ ]:

model.cuda()
model.eval()


In [ ]:
import experiment_thesis.ood.base_prepare
experiment_thesis.ood.base_prepare.OOD_PARAM_SAMPLERS
print("")

In [ ]:
from utils.sampling_strategy import TransformLatentSamplingStrategy

sampling_strategy = TransformLatentSamplingStrategy(transform_seq,clip_data=True)
num_negatives =60
sampler =BatchNegativeSampler(sampling_strategy,number_of_negatives=num_negatives).to(device)


In [ ]:
id_x_mode2, id_y_mode2 = [], []
ood_x_mode2, ood_y_mode2 = [], []

# set random seed (optional — remove if you want non-deterministic randomness)
torch.manual_seed(42)

with torch.no_grad():
    for batch in val_loader:
        x_orig, y_orig = batch
        batch_size = x_orig.size(0)
        M = 1 + num_negatives  # number of candidates per sample

        # Sample positives + negatives (DO NOT change sampler)
        modified_batch = sampler((x_orig.to(device), y_orig.to(device)))
        x_all = modified_batch[0].to(device)   # shape: (batch_size * M, C, H, W)
        y_orig = y_orig.to(device)             # shape: (batch_size,)

        # Forward pass over flattened candidates (batched for memory efficiency)
        for i in range(0, x_all.shape[0], batch_size):
            x_batch = x_all[i:i+batch_size]
            logits_batch = model(x_batch)
            if i == 0:
                logits_all = logits_batch
            else:
                logits_all = torch.cat((logits_all, logits_batch), dim=0)

        # Group back into per-sample candidates: (B, M, num_classes)
        logits_all = torch.stack(logits_all.chunk(M, dim=0), dim=1)
        x_all = torch.stack(x_all.chunk(M, dim=0), dim=1)

        # --- ID Selection: candidate with highest logit for the true class ---
        y_idx = y_orig.view(-1, 1, 1).expand(-1, M, 1)
        true_class_logits = logits_all.gather(2, y_idx).squeeze(2)
        id_indices = torch.argmax(true_class_logits, dim=1)  # (B,)

        # --- OOD Selection: randomly pick another candidate (not the ID one) ---
        all_indices = torch.arange(M, device=device).unsqueeze(0).repeat(batch_size, 1)
        rand_indices = torch.randint(0, M - 1, (batch_size,), device=device)
        # shift indices to avoid id_indices
        ood_indices = torch.where(rand_indices >= id_indices, rand_indices + 1, rand_indices)

        # --- Gather paired samples ---
        ar = torch.arange(batch_size, device=device)
        id_x_batch = x_all[ar, id_indices]
        ood_x_batch = x_all[ar, ood_indices]

        # Append results
        id_x_mode2.append(id_x_batch.cpu())
        id_y_mode2.append(y_orig.cpu())
        ood_x_mode2.append(ood_x_batch.cpu())
        ood_y_mode2.append(y_orig.cpu())

# Concatenate results from all batches
id_x_mode2 = torch.cat(id_x_mode2, dim=0)
id_y_mode2 = torch.cat(id_y_mode2, dim=0)
ood_x_mode2 = torch.cat(ood_x_mode2, dim=0)
ood_y_mode2 = torch.cat(ood_y_mode2, dim=0)

# Build datasets and loaders
dataset_id_mode2 = torch.utils.data.TensorDataset(id_x_mode2, id_y_mode2)
dataset_ood_mode2 = torch.utils.data.TensorDataset(ood_x_mode2, ood_y_mode2)
loader_id_mode2 = torch.utils.data.DataLoader(dataset_id_mode2, batch_size=dataset_info.batch_size, shuffle=False)
loader_ood_mode2 = torch.utils.data.DataLoader(dataset_ood_mode2, batch_size=dataset_info.batch_size, shuffle=False)


In [ ]:
#plot an example from the dataset to see wether they show the same image
#check if data is iamge data
if is_image_data and id_x_mode2.shape[1] in [1,]:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    for i in range(5):
        plt.subplot(2, 5, i + 1)
        plt.imshow(id_x_mode2[i].cpu().squeeze(), cmap='gray')
        plt.title(f"ID Mode 1 - {id_y_mode2[i].item()}")
        plt.axis('off')
        plt.subplot(2, 5, i + 6)
        plt.imshow(ood_x_mode2[i].cpu().squeeze(), cmap='gray')
        plt.title(f"OOD Mode 1 - {id_y_mode2[i].item()}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
from torch.utils.data import DataLoader, Subset

val_dataset = val_loader_transformed.dataset  # assume indexable dataset
n_samples = len(val_dataset)
rng = np.random.default_rng(seed=42)
shuffled_indices = rng.permutation(n_samples)
val_dataset_preshuffled = Subset(val_dataset, shuffled_indices)

val_loader_transformed_preshuffled = DataLoader(
    val_dataset_preshuffled,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers,
)

In [ ]:
n_samples_fraction = int(1/6 * len(val_dataset))
shuffled_indices = rng.permutation(n_samples)[:n_samples_fraction]
val_dataset_preshuffled_small = Subset(val_dataset, shuffled_indices)
val_loader_transformed_preshuffled_small = DataLoader(
    val_dataset_preshuffled_small,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers,
)


In [ ]:
import os
import json
from datetime import datetime
import optuna
import numpy as np
import pandas as pd

from experiment_thesis.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem, run_ood_study_halving,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO

# configure
detector = "knn_mixed"
detector2 = "knn_trap"
ood_objectives = ["auroc", "paired_ood_acc", "fpr95"]
n_trials_ood = 100
n_trials_search_small = 60
n_trials_search_large = 30
num_runs = 4

# Search optimizers
optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0) # For final evaluation
optimizer_search_small = SHGO(initial_samples=10, local_runs=1, local_max_steps=0)
optimizer_search_large = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)




# storage dir for metadata
ood_studies_dir = os.path.join(
    current_path,
    "experiment_files",
    "experiment_hyperparameter_opt_ood",
    dataset,
    architecture,
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(ood_studies_dir, exist_ok=True)

transform_seq_arg = transform_seq

all_results = []

@torch.no_grad()
def run_experiment(
    run_idx,
    exp_name,
    id_loader,
    ood_loader,
    objectives,
    n_trials,
    val_loader_for_search,
    optimizer_for_study=optimizer_search_large,  # do NOT delete this outside object
    use_halving = False,
):
    """Helper to run one full experiment iteration. Memory-leak-aware version."""
    print(f"\n===== Running Experiment: {exp_name} (Run {run_idx + 1}/{num_runs}) =====")
    gc.collect()
    torch.cuda.empty_cache()

    run_dir = os.path.join(ood_studies_dir, exp_name, f"run_{run_idx}")
    os.makedirs(run_dir, exist_ok=True)

    run_results = []

    for objective in objectives:
        train_cache = create_layer_embedding_cache(
            model, train_loader_no_shuffle, cache_config, embedding_cache_path, device=device
        )
        print(f"== Processing objective={objective} ==")
        gc.collect()
        torch.cuda.empty_cache()

        eval_results_path = os.path.join(run_dir, f"{objective}_eval.json")
        params_path = os.path.join(run_dir, f"{objective}_params.json")

        # If both cached files exist, load and continue (no heavy objects created)
        if os.path.exists(eval_results_path) and os.path.exists(params_path):
            print(f"  Found cached results for objective '{objective}'. Loading...")
            with open(eval_results_path, 'r') as f:
                metrics = json.load(f)
            with open(params_path, 'r') as f:
                best_params = json.load(f)

            search_acc = float(metrics["accuracy_mean"])
            search_acc_std = float(metrics["accuracy_std"])
            search_acc_se = float(metrics["accuracy_se"])
            print(f"  Loaded Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")

        else:
            # --- Optimization Step ---
            print(f"  Optimizing {detector} for objective={objective}...")
            study_name = f"ood_{detector}_{objective}_{exp_name}_run{run_idx}"
            storage_path = None  # No DB persistence

            is_direct_search = objective == "search"

            # Build objective kwargs (keep references minimal)
            if is_direct_search:
                report_fraction = 0.1
                if val_loader_for_search == val_loader_transformed_preshuffled_small:
                    report_fraction = 0.2
                    #for coil100 we can use a larger fraction
                    if dataset == "coil100":
                        report_fraction = 0.3
                    if dataset == "tu_berlin":
                        report_fraction = 0.4

                # NOTE: we still pass model/train_cache etc. If run_ood_study stores them internally,
                # it could retain references — so we rely on run_ood_study to not leak. After
                # study completes we will explicitly clear local references.
                objective_kwargs = {
                    "optimizer": optimizer_for_study,
                    "model": model,
                    "train_cache": train_cache,
                    "val_loader": val_loader_for_search,
                    "transform_seq": transform_seq_arg,
                    "dataset_info": dataset_info,
                    "architecture": architecture,
                    "device": str(device),
                    "report_fraction": report_fraction,
                    "repeats": 1,
                }
            else:
                objective_kwargs = {
                    "model": model,
                    "train_cache": train_cache,
                    "id_loader": id_loader,
                    "ood_loader": ood_loader,
                    "transform_seq": transform_seq_arg,
                    "dataset_info": dataset_info,
                    "architecture": architecture,
                    "device": str(device),
                    "metric": objective,
                    "check_percent": 0.1,
                    "prune_at": 0.1,
                    "max_batches": None,
                    "show_progress": False,
                }

            # Run optimization (may be heavy)
            if not use_halving:
                study = run_ood_study(
                    study_name=study_name,
                    storage_path=storage_path,
                    detector_name=detector,
                    objective_type=objective,
                    objective_kwargs=objective_kwargs,
                    n_trials=n_trials,
                )
            else:
                study = run_ood_study_halving(
                    study_name=study_name,
                    storage_path=storage_path,
                    detector_name=detector,
                    objective_type=objective,
                    objective_kwargs=objective_kwargs,
                    n_trials=n_trials,
                )

            # get best params or defaults
            if study is None:
                best_params = get_default_ood_params(detector)
                print(f"  Detector {detector} is parameterless or study skipped; using defaults.")
            else:
                best_params = get_best_ood_params_from_study(study)
                try:
                    print(f"  Best value for {objective}: {study.best_value}")
                except Exception:
                    pass

            # Save best params (convert any numpy/np types to python builtins if necessary)
            with open(params_path, 'w') as f:
                json.dump(json.loads(json.dumps(best_params, default=lambda x: x.tolist() if hasattr(x, "tolist") else str(x))), f, indent=2)
            print(f"  Saved best params to {params_path}")

            # --- Evaluation Step ---
            print(f"  Evaluating final {detector} with best params on search task...")

            # create problem (may keep references to model etc.)
            problem = create_ood_problem(
                detector_name=detector,
                params=best_params,
                model=model,
                train_cache=train_cache,
                transform_seq=transform_seq_arg,
                dataset_info=dataset_info,
                architecture=architecture,
                device=str(device),
            )



            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=optimizer_search_eval,
                problem=problem,
                test_loader=test_loader_transformed,
                save_path=eval_results_path,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=True,
                repeats=3,
                return_per_run=True,  # Get aggregated metrics
                overwrite=True,  # Overwrite eval, not opt study
                store_val=False,
            )

            # Pull out results, convert to python floats
            search_acc = float(metrics["accuracy_mean"])
            search_acc_std = float(metrics["accuracy_std"])
            search_acc_se = float(metrics["accuracy_se"])
            print(f"  Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")

            # print current cuda memory usage
            try:
                print(f"  CUDA Memory Allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB")
            except Exception:
                pass


            study = None
            problem = None
            optimizer_eval = None
            metrics = None
            best_params = None
            gc.collect()
            torch.cuda.empty_cache()

            # After evaluation we saved eval json and params json - reload lightweight dicts
            if os.path.exists(eval_results_path) and os.path.exists(params_path):
                with open(eval_results_path, 'r') as f:
                    metrics = json.load(f)
                with open(params_path, 'r') as f:
                    best_params = json.load(f)
                search_acc = float(metrics["accuracy_mean"])
                search_acc_std = float(metrics["accuracy_std"])
                search_acc_se = float(metrics["accuracy_se"])
                print(f"  Loaded Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")
            else:
                # Fallback if something unexpected happened
                print("  Warning: evaluation files not found after evaluation. Setting metrics to 0.")
                search_acc = 0.0
                search_acc_std = 0.0
                best_params = get_default_ood_params(detector)

        # Final per-objective append (keep best_params small or already json-serializable)
        run_results.append({
            "exp_name": exp_name,
            "run_idx": run_idx,
            "objective": objective,
            "search_accuracy": float(search_acc),
            "search_accuracy_std": float(search_acc_std),
            "search_accuracy_se": float(search_acc_se),
            "best_params": best_params,
        })

        # Clear heavy locals for the next objective
        best_params = None
        metrics = None
        gc.collect()
        torch.cuda.empty_cache()

    return run_results


# --- Experiment 1: OOD tuning on val vs val_transformed ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="val_vs_val_transformed",
        id_loader=val_loader,
        ood_loader=val_loader_transformed,
        objectives=ood_objectives,
        n_trials=n_trials_ood,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)




# --- Experiment 2: OOD tuning on mode2 datasets ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="mode2_id_vs_ood",
        id_loader=loader_id_mode2,
        ood_loader=loader_ood_mode2,
        objectives=ood_objectives,
        n_trials=n_trials_ood,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)
# --- Experiment 3: Direct search optimization (small) ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_small",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_small,
        val_loader_for_search=val_loader_transformed_preshuffled,
        optimizer_for_study=optimizer_search_small,
    )
    all_results.extend(results)
# --- Experiment 4: Direct search optimization (large) ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_large",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_large,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)

# --- Experiment 4: Direct search optimization (large) halfing---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_large_halving",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=60,
        val_loader_for_search=val_loader_transformed_preshuffled,
        use_halving=True,
    )
    all_results.extend(results)

#restricted to small val set to save time
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_val_restricted",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_small,
        val_loader_for_search=val_loader_transformed_preshuffled_small,
    )
    all_results.extend(results)

# --- Experiment 6: Default parameters (no optimization, no objective) ---
for i in range(num_runs):
    print(f"\n===== Running Experiment: default_params (Run {i+1}/{num_runs}) =====")
    gc.collect()
    torch.cuda.empty_cache()

    run_dir = os.path.join(ood_studies_dir, "default_params", f"run_{i}")
    os.makedirs(run_dir, exist_ok=True)

    eval_results_path = os.path.join(run_dir, "default_eval.json")
    params_path = os.path.join(run_dir, "default_params.json")

    if os.path.exists(eval_results_path) and os.path.exists(params_path):
        print("  Found cached default results. Loading...")
        with open(eval_results_path, 'r') as f:
            metrics = json.load(f)
        with open(params_path, 'r') as f:
            best_params = json.load(f)
        search_acc = metrics["accuracy_mean"]
        search_acc_std = metrics["accuracy_std"]
        search_acc_se = metrics["accuracy_se"]

    else:
        # --- Use default parameters ---
        best_params = get_default_ood_params(detector)
        with open(params_path, 'w') as f:
            json.dump(best_params, f, indent=2)
        print(f"  Using default params: {best_params}")

        # --- Evaluation Step ---
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_results_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=True,
            repeats=3,
            return_per_run=True,
            overwrite=True,
            store_val=False,
        )
        search_acc = metrics["accuracy_mean"]
        search_acc_std = metrics["accuracy_std"]
        search_acc_se = metrics["accuracy_se"]

    all_results.append({
        "exp_name": "default_params",
        "run_idx": i,
        "objective": "none",   # mark explicitly that there was no optimization
        "search_accuracy": search_acc,
        "search_accuracy_std": search_acc_std,
        "search_accuracy_se": search_acc_se,
        "best_params": best_params,
    })




print("\n\n" + "="*20 + " FINAL RESULTS SUMMARY " + "="*20)
results_df = pd.DataFrame(all_results)

# Primary metric: mean and std of the mean accuracies across optimization runs
# This captures the variability from the optimization process itself
summary = results_df.groupby(['exp_name', 'objective']).agg({
    'search_accuracy': ['mean', 'std', 'max'],
    'search_accuracy_std': 'mean'  # Also report average evaluation uncertainty
}).reset_index()

# Flatten column names
summary.columns = ['exp_name', 'objective', 'mean_accuracy', 'std_across_runs', 'max_accuracy', 'mean_eval_std']

# Keep numeric version for plotting
summary_numeric = summary.copy()

# Format for display
summary_display = summary.copy()
summary_display['mean_accuracy'] = summary_display['mean_accuracy'].apply(lambda x: f"{x:.4f}")
summary_display['std_across_runs'] = summary_display['std_across_runs'].apply(lambda x: f"{x:.4f}")
summary_display['max_accuracy'] = summary_display['max_accuracy'].apply(lambda x: f"{x:.4f}")
summary_display['mean_eval_std'] = summary_display['mean_eval_std'].apply(lambda x: f"{x:.4f}")

print("\nSummary Statistics:")
print("- mean_accuracy: Average search accuracy across optimization runs")
print("- std_across_runs: Std dev across optimization runs (optimization variability)")
print("- max_accuracy: Best accuracy achieved across all optimization runs")
print("- mean_eval_std: Average evaluation uncertainty within each run")
print()
print(summary_display.to_string(index=False))

# Save detailed results
results_path = os.path.join(ood_studies_dir, "final_experiment_results.json")
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nDetailed results saved to: {results_path}")

# Save summary table
summary_path = os.path.join(ood_studies_dir, "results_summary.csv")
results_df.to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")


In [ ]:
results_df

In [ ]:
#rename experiments val_vs_val_transformed becomes val_vs_transformed
#mode2_id_vs_ood becomes MSP_vs_transformed
#direct_search_val_restricted becomes restricted dataset
#direct_search_small becomes smaller budget
#direct_search_large becomes Default Budget
#default_params becomes No Optimization
results_df["exp_name"] = results_df["exp_name"].replace({
    "val_vs_val_transformed":"Val vs Transformed",
    "mode2_id_vs_ood":"MSP vs transformed",
    "direct_search_val_restricted":"smaller dataset",
    "direct_search_small":"smaller budget",
    "direct_search_large":"default budget",
    "default_params":"no optimization"
})

#ojbective paired_ood_acc becomes Paired Acc
results_df["objective"] = results_df["objective"].replace({
    "paired_ood_acc":"paired acc.",
    "auroc":"AUROC",
    "fpr95":"FPR95",
    "search":"search",
    "none":"none"
})


In [ ]:
results_df
#remove direct_search_large_halving
results_df = results_df[results_df["exp_name"] != "direct_search_large_halving"]

In [ ]:
from utils.eval.vis import plt_setup_latex

In [ ]:
W =plt_setup_latex()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def plot_per_run_results_transposed(results_df, objective_colors=None, figsize=(10, 8)):
    """
    Visualize per-run search accuracies for each experiment and objective (transposed),
    including standard error bars from 'accuracy_se'.
    """

    if objective_colors is None:
        objective_colors = {
            "AUROC": "#1f77b4",          # blue
            "FPR95": "#ff7f0e",          # orange
            "paired acc.": "#2ca02c",    # green
            "search": "#d62728",         # red
            "none": "#7f7f7f",           # gray
        }

    experiments = results_df["exp_name"].unique()
    all_objectives = results_df["objective"].unique()
    n_runs = 4
    bar_height = 0.03

    fig, ax = plt.subplots(figsize=figsize)

    current_y = 0
    y_labels_pos = []
    y_labels = []

    for exp_idx, exp in enumerate(experiments):
        exp_data = results_df[results_df["exp_name"] == exp]
        objectives_in_exp = exp_data["objective"].unique()
        n_objectives_in_exp = len(objectives_in_exp)
        exp_height = n_objectives_in_exp * n_runs * bar_height

        # Plot bars for each objective
        for obj_idx, obj in enumerate(objectives_in_exp):
            obj_data = exp_data[exp_data["objective"] == obj].sort_values("search_accuracy")
            color = objective_colors.get(obj, "#333333")
            accuracies = obj_data["search_accuracy"].values


            se_values = obj_data["search_accuracy_se"].values


            for run_idx in range(len(accuracies)):
                y_pos = current_y + (obj_idx * n_runs * bar_height) + (run_idx * bar_height)
                ax.barh(
                    y_pos,
                    accuracies[run_idx],
                    height=bar_height,
                    color=color,
                    alpha=0.7,
                    edgecolor='black',
                    linewidth=0.5,
                    xerr=se_values[run_idx],  # ← ADD ERROR BARS
                    error_kw=dict(ecolor='black', capsize=1.5, lw=0.6)
                )

        # Center label
        y_labels_pos.append(current_y + exp_height / 2 - bar_height / 2)
        y_labels.append(exp)
        current_y += exp_height + 0.04  # Add vertical gap

    # Set y-axis labels
    ax.set_yticks(y_labels_pos)
    ax.set_yticklabels(y_labels)

    ax.set_xlabel("Accuracy", fontsize=12)
    ax.set_title("Per-Run Search Accuracy Distribution (Transposed)",
                 fontsize=13, fontweight='bold')
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    # Legend
    handles = [plt.Rectangle((0,0), 1, 1, color=objective_colors[obj], alpha=0.8)
               for obj in all_objectives if obj in objective_colors]
    ax.legend(handles, [obj for obj in all_objectives if obj in objective_colors],
              title="Objective", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)

    # Set x-axis limits with padding
    x_min = results_df["search_accuracy"].min() - 0.01
    x_max = results_df["search_accuracy"].max() + 0.01
    ax.set_xlim(x_min, x_max)

    plt.tight_layout()
    path = os.path.join(current_path, "experiment_files", "export", "results", "hyper_opt", dataset, transform_name)
    os.makedirs(path, exist_ok=True)
    plt.savefig(path + 'per_run_distributions_transposed.png', dpi=300, bbox_inches='tight')
    plt.savefig(path + 'per_run_distributions_transposed.pgf', bbox_inches='tight')
    plt.savefig(path + 'per_run_distributions_transposed.pdf', bbox_inches='tight')
    plt.show()
    print("Plot saved as 'per_run_distributions_transposed.png'")


# Example usage:
plot_per_run_results_transposed(results_df, figsize=(W, W))


In [ ]:
summary["max_accuracy"]

In [ ]:
stop

In [ ]:
print(torch.__file__)

In [ ]:
import torch, gc, psutil, time

def get_gpu_mem():
    """Return allocated and reserved GPU memory in MB."""
    return torch.cuda.memory_allocated() / 1024**2, torch.cuda.memory_reserved() / 1024**2

def get_cpu_mem():
    """Return current process CPU memory in MB."""
    return psutil.Process().memory_info().rss / 1024**2

def print_state(tag):
    alloc, res = get_gpu_mem()
    print(f"[{tag}] GPU allocated={alloc:.2f} MB | reserved={res:.2f} MB | CPU={get_cpu_mem():.2f} MB")

def cleanup_step(name):
    """Force garbage collection and clear CUDA cache."""
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(0.01)
    print_state(f"After deleting {name}")

def interactive_cleanup(exclude_prefixes=("torch", "gc", "psutil", "time", "get_", "print_", "cleanup_", "interactive_cleanup")):
    """
    Iteratively delete variables from globals(), observing memory impact.
    Keeps the cleanup helpers themselves alive.
    """
    print_state("Initial")
    candidates = [k for k in globals().keys()
                  if not k.startswith("_")
                  and not any(k.startswith(pref) for pref in exclude_prefixes)
                  and k not in ("interactive_cleanup",)]

    if not candidates:
        print("No user variables to test.")
        return

    print(f"\nFound {len(candidates)} user variables to test:\n  {candidates}\n")
    for name in candidates:
        obj = globals().get(name)
        size_info = ""
        try:
            if torch.is_tensor(obj):
                size_info = f"(Tensor {tuple(obj.shape)} | {obj.element_size() * obj.nelement() / 1024**2:.2f} MB)"
            elif isinstance(obj, (list, dict, tuple)):
                size_info = f"(len={len(obj)})"
            elif hasattr(obj, "__class__"):
                size_info = f"({obj.__class__.__name__})"
        except Exception:
            pass

        print(f"→ Deleting {name} {size_info}")
        del globals()[name]
        cleanup_step(name)

    print("\n🧹 All user variables tested.")
    print("Now checking whether clearing modules changes memory...")

    import sys
    user_mods = [m for m in list(sys.modules.keys()) if "experiments_rotation" in m]
    for m in user_mods:
        print(f"→ Deleting module: {m}")
        del sys.modules[m]
        cleanup_step(f"module {m}")

    print("\n✅ Finished step-by-step cleanup. (Remaining allocations may persist due to CUDA context.)")

# Run it
interactive_cleanup()


In [ ]:
#print memory allocation
print(torch.cuda.memory_summary())
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(torch.cuda.memory_allocated() / 1024**2)  # MB actually used by tensors
print(torch.cuda.memory_reserved() / 1024**2)   # MB reserved by caching allocator

In [ ]:
import torch, gc
from confidence.utils import ModelInputOutputWrapper

# Collect garbage first
gc.collect()

# List all tensors still alive
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            #if tensor has shape [1] print its value
            if obj.numel() == 1:
                print(obj.item(), obj.size(), obj.dtype, type(obj))
            else:
                print(obj.size(), obj.dtype, type(obj))
        if isinstance(obj, ModelInputOutputWrapper):
            print("ModelInputOutputWrapper:", obj)

    except Exception:
        pass


In [ ]:
#print hooks on main model
for name, module in model.named_modules():
    if hasattr(module, "_forward_hooks") and module._forward_hooks:
        print(f"Module: {name}, Hooks: {module._forward_hooks}")

In [ ]:
print("Hello World")


In [ ]:
print("Hello World")



In [ ]:
print("Hello World")


In [ ]:

print("Hello World")

In [ ]:
%reset out